In [3]:
import time
from labjack import ljm

# --- Configuration Constants ---
DEVICE_TYPE = "T7"
CONNECTION_TYPE = "ANY"  # "USB" or "ETHERNET"
IDENTIFIER = "ANY"

# Hardware Channels
DAC_CHANNEL = "TDAC2"     # LJTick-DAC A on FIO2 (DIO2)
AIN_CHANNEL = "AIN5"      # Current feedback read pin

# Shunt & Scaling parameters
SHUNT_RESISTOR = 5.9      # ohms
GAIN = 20.0               # LJTCS amplifier gain
FLOW_MIN_MA = 4.0
FLOW_MAX_MA = 20.0
FLOW_MIN_SCCM = 0.0
FLOW_MAX_SCCM = 200.0

def calculate_current_ma(voltage):
    """Converts the raw AIN voltage from the LJTCS into mA."""
    # V = I * R * Gain -> I(A) = V / (R * Gain)
    current_amps = voltage / (SHUNT_RESISTOR * GAIN)
    return current_amps * 1000.0

def calculate_flowrate(current_ma):
    """Scales a 4-20mA signal to 0-200 sccm."""
    # Clamp current to the active 4-20mA scaling range
    clamped_current = max(FLOW_MIN_MA, min(current_ma, FLOW_MAX_MA))
    
    # Linear interpolation: y = y0 + (x - x0) * (y1 - y0) / (x1 - x0)
    flowrate = FLOW_MIN_SCCM + (clamped_current - FLOW_MIN_MA) * (
        (FLOW_MAX_SCCM - FLOW_MIN_SCCM) / (FLOW_MAX_MA - FLOW_MIN_MA)
    )
    return flowrate

def main():
    handle = None
    try:
        # Open connection to the LabJack T7
        handle = ljm.openS(DEVICE_TYPE, CONNECTION_TYPE, IDENTIFIER)
        
        # Define our sweep profile (0V to 5V in steps of 0.5V)
        voltages_to_test = [v * 0.5 for v in range(11)]
        
        print(f"{'Set DAC (V)':<12} | {'Read AIN5 (V)':<14} | {'Current (mA)':<13} | {'Flowrate (sccm)':<15}")
        print("-" * 65)
        
        for set_v in voltages_to_test:
            # 1. Output control voltage to the gas flow controller
            ljm.eWriteName(handle, DAC_CHANNEL, set_v)
            
            # 2. Settle time (allowing the physical controller to respond)
            time.sleep(0.8)
            
            # 3. Read the feedback voltage
            read_v = ljm.eReadName(handle, AIN_CHANNEL)
            
            # 4. Process electrical and physical metrics
            current_ma = calculate_current_ma(read_v)
            flow_sccm = calculate_flowrate(current_ma)
            
            # Display step data
            print(f"{set_v:<12.2f} | {read_v:<14.4f} | {current_ma:<13.3f} | {flow_sccm:<15.2f}")
            
    except ljm.LJMError as e:
        print(f"\nLJM Library Error: {e}")
    except Exception as e:
        print(f"\nUnexpected Error: {e}")
    finally:
        if handle is not None:
            # Safe shutdown: Set control output back to 0V before closing
            try:
                ljm.eWriteName(handle, DAC_CHANNEL, 0.0)
                print("\nOutput reset to 0.0V. Connection safely closed.")
            except Exception:
                pass
            ljm.close(handle)

if __name__ == "__main__":
    main()

Set DAC (V)  | Read AIN5 (V)  | Current (mA)  | Flowrate (sccm)
-----------------------------------------------------------------
0.00         | 0.5453         | 4.621         | 7.77           
0.50         | 0.5455         | 4.623         | 7.78           
1.00         | 0.5455         | 4.623         | 7.78           
1.50         | 0.5460         | 4.627         | 7.83           
2.00         | 1.2082         | 10.239        | 77.99          
2.50         | 1.4354         | 12.165        | 102.06         
3.00         | 1.6250         | 13.771        | 122.14         
3.50         | 1.8168         | 15.396        | 142.45         
4.00         | 2.0071         | 17.010        | 162.62         
4.50         | 2.1942         | 18.595        | 182.44         
5.00         | 2.3598         | 19.998        | 199.98         

Output reset to 0.0V. Connection safely closed.
